In [ ]:
"""
Test script for the Truthify FastAPI
This script demonstrates how to use the API endpoints for claim extraction and fact-checking.
"""

import json
import time

import requests

# API base URL (adjust if running on different host/port)
BASE_URL = "http://localhost:8000"


def test_health_check():
    """Test the health check endpoint"""
    print("🔍 Testing health check endpoint...")
    response = requests.get(f"{BASE_URL}/health")
    print(f"Status: {response.status_code}")
    print(f"Response: {json.dumps(response.json(), indent=2)}")
    print("-" * 60)


def test_extract_claims(text):
    """Test claim extraction without fact-checking"""
    print("🔍 Testing claim extraction endpoint...")
    print(f"Input text: {text}")

    payload = {"text": text}
    response = requests.post(f"{BASE_URL}/extract-claims", json=payload)

    print(f"Status: {response.status_code}")
    if response.status_code == 200:
        result = response.json()
        print(f"Processing time: {result['processing_time']:.2f}s")
        print(f"Extracted claims ({len(result['extracted_claims'])}):")
        for i, claim in enumerate(result["extracted_claims"], 1):
            print(f"  {i}. {claim}")
    else:
        print(f"Error: {response.text}")
    print("-" * 60)


def test_fact_check(text):
    """Test full fact-checking pipeline"""
    print("🔍 Testing fact-checking endpoint...")
    print(f"Input text: {text}")

    payload = {"text": text}
    response = requests.post(f"{BASE_URL}/fact-check", json=payload)

    print(f"Status: {response.status_code}")
    if response.status_code == 200:
        result = response.json()
        print(f"Processing time: {result['processing_time']:.2f}s")
        print(f"Extracted claims ({len(result['extracted_claims'])}):")
        for i, claim in enumerate(result["extracted_claims"], 1):
            print(f"  {i}. {claim}")

        print(f"\nFact-check results:")
        for i, fact_result in enumerate(result["fact_check_results"], 1):
            print(f"\n🔹 Claim {i}: {fact_result['claim']}")
            print(f"Status: {fact_result['status']}")
            print(f"Explanation: {fact_result['explanation']}")
            if fact_result["sources"]:
                print("Sources:")
                for j, source in enumerate(fact_result["sources"], 1):
                    print(f"  {j}. {source['title']} - {source['url']}")
            else:
                print("No sources available.")
    else:
        print(f"Error: {response.text}")
    print("-" * 60)


def main():
    """Main test function"""
    print("🚀 Starting Truthify API Tests\n")

    # Test health check
    try:
        test_health_check()
    except requests.exceptions.ConnectionError:
        print("❌ Cannot connect to API. Make sure the server is running on http://localhost:8000")
        print("To start the server, run: python src/api.py")
        return

    # Sample texts for testing
    test_texts = [
        "The Earth is flat and vaccines cause autism.",
        "In my opinion, the sky is blue and water is wet.",
        "COVID-19 vaccines were developed in record time and have been proven safe and effective.",
    ]

    # Test claim extraction for each text
    for i, text in enumerate(test_texts, 1):
        print(f"\n📝 Test Case {i} - Claim Extraction")
        test_extract_claims(text)

    # Test full fact-checking for one example (since it takes longer)
    print(f"\n📝 Full Fact-Check Test")
    test_fact_check(test_texts[0])


if __name__ == "__main__":
    main()
